In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/photoprotein/D7PM12.fasta.txt
/kaggle/input/photoprotein/train_seqs-30_cluster.tsv
/kaggle/input/photoprotein/mluc2.txt
/kaggle/input/photoprotein/test_photopr.csv
/kaggle/input/photoprotein/P42212.fasta.txt
/kaggle/input/photoprotein/train_photoprotein.fasta
/kaggle/input/photoprotein/saved_weights_esm_fine_tune
/kaggle/input/photoprotein/half1/half1/Q95Q68_C37H5.13.pt
/kaggle/input/photoprotein/half1/half1/R4YUH5_OLEAN_C36970.pt
/kaggle/input/photoprotein/half1/half1/R4YUW4_OLEAN_C29780.pt
/kaggle/input/photoprotein/half1/half1/R4YMC0_ftsE.pt
/kaggle/input/photoprotein/half1/half1/Q9I8Y1_adarb1a.pt
/kaggle/input/photoprotein/half1/half1/P0A812_ruvB.pt
/kaggle/input/photoprotein/half1/half1/Q9LKR3_MED37A.pt
/kaggle/input/photoprotein/half1/half1/P35164_resE.pt
/kaggle/input/photoprotein/half1/half1/O32223_iolW.pt
/kaggle/input/photoprotein/half1/half1/Q5SIL8.pt
/kaggle/input/photoprotein/half1/half1/Q93VK5_CYP97A3.pt
/kaggle/input/photoprotein/half1/half1/Q9D787_Ppil2.pt

In [2]:
import os
import torch
import numpy as np
from  tqdm import tqdm
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from scipy import stats
from sklearn.metrics import mean_absolute_error

In [3]:
need_dir = "/kaggle/input/photoprotein/half1/half1"

In [4]:
def process_tensor(mean_repr):
    three = mean_repr[33].reshape(1, -1)
    three = three.numpy()
    return three

In [5]:
content = os.listdir(need_dir)

In [6]:
X = []
labels = []
for data in tqdm(content):
    path = os.path.join(need_dir, data)
    embed = torch.load(path)
    mean_repr = embed["mean_representations"]
    average_embed = process_tensor(mean_repr)
    X.append(average_embed)
    labels.append(embed["label"])

100%|██████████| 14503/14503 [17:11<00:00, 14.06it/s]


In [7]:
def create_df(path):
    with open(path) as fi:
        data = fi.read()
    data = data.split("\n")
    prot_id = []
    seqs = []
    for i in range(1, len(data), 2):
        prot_id.append(data[i-1].replace(">", ""))
        seqs.append(data[i])
    df = {"Protein ID": prot_id, "simple_fasta": seqs}
    df = pd.DataFrame(df)
    return df

In [8]:
df1 = create_df('/kaggle/input/tm-predcition/Hyperthermophiliccd_hit_out')
df2 = create_df('/kaggle/input/tm-predcition/Mesophiliccd_hit_out')
df3 = create_df('/kaggle/input/tm-predcition/Thermophiliccd_hit_out')
    

In [9]:
frames = [df1, df2, df3]
df = pd.concat(frames)

In [10]:
tm_per_seq = pd.read_csv("/kaggle/input/tm-predcition/filtered_data.csv")

In [11]:
filtered_seqs = pd.merge(df, tm_per_seq, how ='inner', on = ['Protein ID', 'simple_fasta'])

In [12]:
embed = [list(i) for i in X]
d = {"embed": embed, "Protein ID": labels}
emb_per_tm = pd.DataFrame(d)

In [13]:
filtered_seqs = pd.merge(filtered_seqs, emb_per_tm, how ='inner', on = ['Protein ID'])

In [14]:
features = np.array(list(map(lambda x: x, filtered_seqs.embed)))
features = features.reshape(features.shape[0], -1)
target = np.array(list(map(lambda x: x, filtered_seqs.Tm)))

In [15]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.30, random_state=42)


In [16]:
my_model = XGBRegressor(n_estimators=1000, learning_rate=0.05, n_jobs=4, early_stopping_rounds=5,)
my_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], 
             verbose=False)

XGBRegressor(base_score=0.5, booster='gbtree', callbacks=None,
             colsample_bylevel=1, colsample_bynode=1, colsample_bytree=1,
             early_stopping_rounds=5, enable_categorical=False,
             eval_metric=None, gamma=0, gpu_id=-1, grow_policy='depthwise',
             importance_type=None, interaction_constraints='',
             learning_rate=0.05, max_bin=256, max_cat_to_onehot=4,
             max_delta_step=0, max_depth=6, max_leaves=0, min_child_weight=1,
             missing=nan, monotone_constraints='()', n_estimators=1000,
             n_jobs=4, num_parallel_tree=1, predictor='auto', random_state=0,
             reg_alpha=0, reg_lambda=1, ...)

In [17]:
predictions = my_model.predict(X_test)
mse = mean_squared_error(y_test, predictions)
rmse = mean_squared_error(y_test, predictions, squared=False)
r_2 = r2_score(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)
pcc, _ = stats.pearsonr(y_test, predictions)

In [18]:
def print_metrics(**kwargs):
    metrics = pd.DataFrame(kwargs, index = [0])
    return metrics

In [19]:
print_metrics(mse = mse, mae = mae, rmse = rmse, r_2 = r_2, pcc = pcc)

,mse,mae,rmse,r_2,pcc
0,52.758085,5.410317,7.263476,0.666189,0.818536
